# Performance Optimization & Scaling — dask_setup Recipe

This notebook covers the three main levers for tuning `dask_setup` performance:

1. **Worker count scaling** — speedup and efficiency vs. number of workers
2. **Workload type comparison** — cpu vs. io vs. mixed for different task mixes
3. **Chunk size impact** — how chunk size affects throughput and memory
4. **Using the built-in benchmarking API** (`benchmark_config`, `scaling_analysis`, `chunk_impact`)

## Setup

In [ ]:
import time

import dask.array as da
import numpy as np
import xarray as xr

from dask_setup import setup_dask_client
from dask_setup.config import DaskSetupConfig
from dask_setup.benchmark import (
    benchmark_config,
    scaling_analysis,
    chunk_impact,
    run_synthetic_benchmark,
)

try:
    import matplotlib.pyplot as plt
    PLOT = True
except ImportError:
    PLOT = False
    print("matplotlib not found — plots disabled")

print("Imports OK")

## 1. Quick Synthetic Benchmark

`run_synthetic_benchmark()` runs without any real dataset — perfect for quickly
checking how a profile performs on your machine.

In [ ]:
result = run_synthetic_benchmark(
    profile_name="development",
    operation="mean",
    ds_size="small",
    repeats=2,
    verbose=True,
)
print()
print(result.summary())

## 2. A/B-Test Configurations with `benchmark_config()`

Compare multiple `DaskSetupConfig` objects on the **same** xarray dataset.
Each config gets a fresh cluster so results are directly comparable.

In [ ]:
# Build a small synthetic xarray dataset for benchmarking
np.random.seed(0)
arr = np.random.random((300, 90, 180)).astype("float32")
bench_ds = xr.Dataset(
    {"temperature": (["time", "lat", "lon"], arr)},
    coords={
        "time": np.arange(300),
        "lat":  np.linspace(-90, 90, 90),
        "lon":  np.linspace(-180, 180, 180),
    },
)
print(f"Dataset: {dict(bench_ds.dims)}  size={bench_ds.nbytes / 1e6:.1f} MB")

In [ ]:
configs = {
    "cpu_2w":   DaskSetupConfig(workload_type="cpu",   max_workers=2, reserve_mem_gb=2.0),
    "io_1w":    DaskSetupConfig(workload_type="io",    max_workers=1, reserve_mem_gb=2.0),
    "mixed_2w": DaskSetupConfig(workload_type="mixed", max_workers=2, reserve_mem_gb=2.0),
}

results = benchmark_config(
    configs,
    bench_ds,
    operation="mean",
    repeats=2,
    verbose=True,
)

print("\nRanked by wall time:")
for r in sorted(results, key=lambda r: r.wall_time_seconds):
    print(f"  {r.summary_line()}")

## 3. Worker Count Scaling

`scaling_analysis()` sweeps worker counts and reports speedup and parallel efficiency
relative to the single-worker baseline.

In [ ]:
scaling = scaling_analysis(
    bench_ds,
    operation="mean",
    worker_counts=(1, 2),   # keep it short for the demo; add 4, 8 on real HPC
    base_config=DaskSetupConfig(
        workload_type="cpu",
        reserve_mem_gb=2.0,
        fallback_on_detection_failure=True,
    ),
    repeats=2,
    verbose=True,
)

print()
print(scaling.summary())
print(f"\nBest: {scaling.best().summary_line()}")

In [ ]:
if PLOT:
    fig = scaling.plot(title="Scaling analysis — synthetic xarray dataset")
    if fig:
        plt.show()

## 4. Chunk Size Impact

`chunk_impact()` uses a **fixed running cluster** and sweeps chunk sizes to find the
throughput sweet spot for a specific dataset and operation.

In [ ]:
client, cluster, tmp = setup_dask_client(
    workload_type="cpu",
    max_workers=2,
    reserve_mem_gb=2.0,
    fallback_on_detection_failure=True,
    dashboard=False,
)

impact = chunk_impact(
    bench_ds,
    client,
    operation="mean",
    chunk_sizes=[
        {"time": 30,  "lat": 45, "lon": 90},
        {"time": 60,  "lat": 90, "lon": 180},
        {"time": 150, "lat": 90, "lon": 180},
        {"time": 300, "lat": 90, "lon": 180},
    ],
    repeats=2,
    verbose=True,
)

print()
print(impact.summary())

In [ ]:
if PLOT:
    fig = impact.plot(dim="time", title="Chunk size impact — time dimension")
    if fig:
        plt.show()

In [ ]:
client.close()
cluster.close()
print("Cluster closed.")

## 5. Exporting Results

`ScalingResult` and `ChunkImpactResult` both have a `.to_dataframe()` method for easy
export to CSV or further analysis.

In [ ]:
try:
    df_scaling = scaling.to_dataframe()
    print("Scaling results:")
    print(df_scaling[["workers", "wall_time_s", "speedup", "efficiency"]].to_string(index=False))
except ImportError:
    print("pandas not installed — .to_dataframe() requires pandas")

## Performance Tuning Checklist

1. **Choose the right `workload_type`** — `cpu` for maths, `io` for file loading, `mixed` for both.
2. **Hit the 256–512 MiB chunk target** — use `recommend_chunks()` or `chunk_impact()` to guide this.
3. **Check parallel efficiency** — if `scaling_analysis()` shows <70% efficiency at 4 workers, the bottleneck is not compute (check I/O, shuffle, or task graph overhead).
4. **Enable spill compression** — `spill_compression="lz4"` reduces spill latency on SSDs.
5. **Use `adaptive_memory=True`** for jobs that start cold — it tightens memory thresholds automatically after the first minute.

See the [Benchmarking wiki page](https://github.com/21centuryweather/dask_setup/wiki/Benchmarking) for full API documentation.